# 02 — WKNN Localization
## MR Localization via Radio Fingerprinting

**Purpose.** This notebook constitutes Module 2 (M2) of the localization pipeline. It consumes the fingerprint database built in M1 (`fp_db.parquet`) and the pre-split measurement records from M0 (`measurement_data_train.parquet`, `measurement_data_test.parquet`), reconstructs simulated MR query events, and applies Weighted K-Nearest Neighbors (WKNN) matching to estimate the geographic position of each query. The output (`mr_located.parquet`) is the input for M3 (evaluation) and M4 (visualization).

**Method.** Each query event is a `(ue_id, date)` group representing one simulated Measurement Report — a sparse vector of observed RSRP values across up to 26 cells. Position is estimated as the inverse-distance-weighted centroid of the K nearest reference fingerprints in Euclidean RSRP space. Cells not observed in a query are filled with −120 dBm (noise floor), matching the fill convention used to build `fp_db`.

**Inputs.**

| File | Source | Description |
|---|---|---|
| `data/processed/fp_db.parquet` | M1 | Fingerprint database — training positions × 28 columns (`sim_x`, `sim_y`, 26 cell RSRP) |
| `data/processed/measurement_data_train.parquet` | M0 | Training UE records (long-format) |
| `data/processed/measurement_data_test.parquet` | M0 | Test UE records (long-format) |

**Notebook structure.**

| Section | Topic |
|---|---|
| §1 | Load Inputs |
| §2 | Query Event Reconstruction |
| §3 | Query Vector Construction |
| §4 | WKNN Model |
| §5 | Localization |
| §6 | Sanity Check |
| §7 | Save Outputs |

**Output.** `data/processed/mr_located.parquet` — one row per query event with true and estimated positions.

## 0. Setup

In [1]:
import os
import warnings

import numpy as np
import pandas as pd
from sklearn.neighbors import KNeighborsRegressor

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 20)
pd.set_option('display.float_format', '{:.3f}'.format)

PROCESSED_DIR = '../data/processed'
FP_DB_PATH    = f'{PROCESSED_DIR}/fp_db.parquet'
TRAIN_PATH    = f'{PROCESSED_DIR}/measurement_data_train.parquet'
TEST_PATH     = f'{PROCESSED_DIR}/measurement_data_test.parquet'
FILL_VAL      = -120.0
K             = 5

print('Libraries loaded.')

Libraries loaded.


## 1. Load Inputs

Load `fp_db.parquet` as the reference set and the pre-split `measurement_data_train.parquet` / `measurement_data_test.parquet` as sources for query event reconstruction. Verify that the cell columns in `fp_db` are consistent with the cells present in both splits.

| Variable | Description |
|---|---|
| `fp_db` | Fingerprint DB: `sim_x`, `sim_y`, + 26 cell columns |
| `train_df` | Long-format training records |
| `test_df` | Long-format test records |
| `cell_cols` | The 26 `gcell_id` column names shared by all datasets |

In [2]:
fp_db    = pd.read_parquet(FP_DB_PATH)
train_df = pd.read_parquet(TRAIN_PATH)
test_df  = pd.read_parquet(TEST_PATH)

cell_cols = [c for c in fp_db.columns if c not in ['sim_x', 'sim_y']]
X_fp      = fp_db[cell_cols].values
y_fp      = fp_db[['sim_x', 'sim_y']].values

print(f'fp_db       : {fp_db.shape}  ({len(fp_db):,} positions x {len(cell_cols)} cells + 2 coords)')
print(f'train_df    : {train_df.shape}  ({train_df["ue_id"].nunique():,} UEs)')
print(f'test_df     : {test_df.shape}  ({test_df["ue_id"].nunique():,} UEs)')
print(f'UE overlap  : {len(set(train_df["ue_id"]) & set(test_df["ue_id"]))}  (must be 0)')

# Verify cell columns align between fp_db and both measurement splits
meas_cells = set(pd.concat([train_df, test_df])['gcell_id'].unique())
fp_cells   = set(cell_cols)
print(f'\nCell alignment check:')
print(f'  fp_db cells: {len(fp_cells)},  measurement cells: {len(meas_cells)}')
print(f'  Symmetric difference: {len(fp_cells.symmetric_difference(meas_cells))}  (must be 0)')

fp_db       : (6746, 28)  (6,746 positions x 26 cells + 2 coords)
train_df    : (34022, 7)  (1,192 UEs)
test_df     : (7453, 7)  (298 UEs)
UE overlap  : 0  (must be 0)

Cell alignment check:
  fp_db cells: 26,  measurement cells: 26
  Symmetric difference: 0  (must be 0)


## 2. Query Event Reconstruction

In the raw data, each simulated MR is stored as multiple rows — one per cell observed during that `(ue_id, date)` timestamp. Grouping by `(ue_id, date)` reconstructs a single query event: a set of `{gcell_id → rsrp}` pairs analogous to a real Measurement Report.

This section builds `query_train` and `query_test` — long-format tables where each group is one MR — and attaches the true `(sim_x, sim_y)` coordinates that serve as ground truth in M3.

| Variable | Description |
|---|---|
| `query_train` | Long-format training events: one row per `(ue_id, date, gcell_id)` triple |
| `query_test`  | Long-format test events: one row per `(ue_id, date, gcell_id)` triple |
| `n_train_events` | Number of unique training `(ue_id, date)` query events |
| `n_test_events`  | Number of unique test `(ue_id, date)` query events |

In [3]:
# The measurement parquets are already the query event source.
# Add a 'split' label and compute event counts.
query_train = train_df.copy()
query_test  = test_df.copy()

n_train_events = query_train.groupby(['ue_id', 'date']).ngroups
n_test_events  = query_test.groupby(['ue_id', 'date']).ngroups

print(f'Training query events : {n_train_events:,}')
print(f'Test query events     : {n_test_events:,}')
print(f'Total events          : {n_train_events + n_test_events:,}')

# Verify positional uniqueness within events (UE does not move within one MR)
for label, df in [('train', query_train), ('test', query_test)]:
    multi_pos = (df.groupby(['ue_id', 'date'])[['sim_x', 'sim_y']].nunique() > 1).any(axis=1).sum()
    print(f'Events with ambiguous position ({label}): {multi_pos}  (must be 0)')

Training query events : 6,397
Test query events     : 1,522
Total events          : 7,919
Events with ambiguous position (train): 3533  (must be 0)
Events with ambiguous position (test): 852  (must be 0)


## 3. Query Vector Construction

Transform each query event into a 26-dimensional RSRP vector matching the column schema of `fp_db`. The pivot mirrors the M1 logic: group by `(ue_id, date)`, unstack `gcell_id` as columns, aggregate with mean, then fill cells not observed in the query with `−120.0` dBm.

The resulting matrices `X_train` / `X_test` have identical column order to `fp_db[cell_cols]`. The target arrays `y_train` / `y_test` hold the corresponding true `(sim_x, sim_y)` positions.

| Variable | Shape | Description |
|---|---|---|
| `X_train` | (n_train_events, 26) | Query RSRP vectors for train events |
| `X_test`  | (n_test_events, 26)  | Query RSRP vectors for test events |
| `y_train` | (n_train_events, 2)  | True `(sim_x, sim_y)` for train events |
| `y_test`  | (n_test_events, 2)   | True `(sim_x, sim_y)` for test events |

In [4]:
def build_query_matrix(df, cell_cols, fill_val=FILL_VAL):
    """Pivot long-format MR records to wide RSRP matrix aligned to cell_cols."""
    # True positions per event (constant within event)
    true_pos = (df.groupby(['ue_id', 'date'])[['sim_x', 'sim_y']]
                .first().reset_index())
    # Pivot: (ue_id, date) x gcell_id -> mean rsrp
    pivot = (df.groupby(['ue_id', 'date', 'gcell_id'])['rsrp']
             .mean().unstack('gcell_id').reset_index())
    # Align to fp_db column order, fill missing
    for col in cell_cols:
        if col not in pivot.columns:
            pivot[col] = fill_val
    pivot = pivot.fillna(fill_val)
    # Merge with true positions
    merged = true_pos.merge(pivot[['ue_id', 'date'] + cell_cols], on=['ue_id', 'date'])
    X = merged[cell_cols].values
    y = merged[['sim_x', 'sim_y']].values
    meta = merged[['ue_id', 'date']].copy()
    return X, y, meta

X_train, y_train, meta_train = build_query_matrix(query_train, cell_cols)
X_test,  y_test,  meta_test  = build_query_matrix(query_test,  cell_cols)

print(f'X_train: {X_train.shape},  y_train: {y_train.shape}')
print(f'X_test : {X_test.shape},   y_test : {y_test.shape}')
print(f'X_fp   : {X_fp.shape},     y_fp   : {y_fp.shape}')

X_train: (6397, 26),  y_train: (6397, 2)
X_test : (1522, 26),   y_test : (1522, 2)
X_fp   : (6746, 26),     y_fp   : (6746, 2)


## 4. WKNN Model

Fit a `KNeighborsRegressor` from scikit-learn on the full fingerprint database.

| Parameter | Value | Rationale |
|---|---|---|
| `n_neighbors` | 5 (default; sweep in M3) | Primary hyperparameter; M3 evaluates K ∈ {1, 3, 5, 7, 9, 15} |
| `weights` | `'distance'` | Inverse-distance weighting — closer fingerprints contribute more |
| `metric` | `'euclidean'` | Euclidean distance in 26-D RSRP space |
| `algorithm` | `'auto'` | scikit-learn selects kd-tree or ball-tree based on data size |

The model is fit on `(X_fp, y_fp)` where `y_fp` is a 2-column matrix `[sim_x, sim_y]`.

In [5]:
wknn = KNeighborsRegressor(n_neighbors=K, weights='distance', metric='euclidean', algorithm='auto')
wknn.fit(X_fp, y_fp)
print(f'WKNN fitted on {len(X_fp):,} fingerprints  (K={K}, weights=distance, metric=euclidean)')

WKNN fitted on 6,746 fingerprints  (K=5, weights=distance, metric=euclidean)


## 5. Localization

Apply the fitted WKNN model to `X_train` and `X_test`. For each query vector, the model returns the inverse-distance-weighted centroid `(est_x, est_y)` of its K nearest fingerprints.

Combine predictions with true positions and event metadata into `mr_located`:

| Column | Description |
|---|---|
| `ue_id` | UE identifier |
| `date` | Measurement timestamp (event key) |
| `true_x`, `true_y` | Ground-truth position |
| `est_x`, `est_y` | WKNN position estimate |
| `split` | `'train'` or `'test'` |
| `error_m` | Euclidean localization error in metres |

In [6]:
pred_train = wknn.predict(X_train)
pred_test  = wknn.predict(X_test)

def make_result_df(meta, y_true, y_pred, split_label):
    df = meta.copy()
    df['true_x'] = y_true[:, 0]
    df['true_y'] = y_true[:, 1]
    df['est_x']  = y_pred[:, 0]
    df['est_y']  = y_pred[:, 1]
    df['split']  = split_label
    df['error_m'] = np.sqrt((df['est_x'] - df['true_x'])**2 +
                            (df['est_y'] - df['true_y'])**2)
    return df

mr_train   = make_result_df(meta_train, y_train, pred_train, 'train')
mr_test    = make_result_df(meta_test,  y_test,  pred_test,  'test')
mr_located = pd.concat([mr_train, mr_test], ignore_index=True)

print(f'mr_located: {mr_located.shape}  (train: {len(mr_train)}, test: {len(mr_test)})')
print(f'\nTest split summary:')
print(f'  Mean error   : {mr_test["error_m"].mean():.1f} m')
print(f'  Median error : {mr_test["error_m"].median():.1f} m')
mr_located.head(3)

mr_located: (7919, 8)  (train: 6397, test: 1522)

Test split summary:
  Mean error   : 502.1 m
  Median error : 174.7 m


,ue_id,date,true_x,true_y,est_x,est_y,split,error_m
0,005c15eb9f4d54169e0faad8f339b33d,2026-03-20 04:27:11.980000+00:00,-62.982,-873.690,-231.784,-924.515,train,176.287
1,005c15eb9f4d54169e0faad8f339b33d,2026-03-20 04:27:12.006000+00:00,-62.982,-873.690,-118.803,-916.999,train,70.651
2,005d1401ecd15f1f830c51d30f4e487b,2026-03-13 11:13:13.571000+00:00,-19.269,-134.552,-22.631,-132.762,train,3.809


## 6. Sanity Check

Validate before persisting:

1. **Coordinate bounds** — all `est_x` ∈ [−1212, −145] m and `est_y` ∈ [−662, 491] m (study area bounding box)
2. **No NaNs** — confirm `est_x`, `est_y`, `true_x`, `true_y` are fully populated
3. **Row count** — `len(mr_located) == n_train_events + n_test_events`
4. **Error distribution preview** — mean and median error for test split

In [7]:
# 1. Coordinate bounds
print('Coordinate bounds:')
for col, lo, hi in [('est_x', -1212, -145), ('est_y', -662, 491)]:
    out = mr_located[(mr_located[col] < lo) | (mr_located[col] > hi)]
    print(f'  {col}: [{mr_located[col].min():.1f}, {mr_located[col].max():.1f}]  '
          f'| out-of-bounds: {len(out)}')

# 2. NaN check
nan_counts = mr_located[['est_x', 'est_y', 'true_x', 'true_y']].isna().sum()
print(f'\nNaN counts:\n{nan_counts.to_string()}')

# 3. Row count
expected = n_train_events + n_test_events
print(f'\nRow count: {len(mr_located)} (expected {expected})  OK={len(mr_located)==expected}')

# 4. Error distribution
test_errors = mr_located[mr_located['split'] == 'test']['error_m']
print(f'\nTest error distribution (K={K}):')
print(f'  Mean   : {test_errors.mean():.1f} m')
print(f'  Median : {test_errors.median():.1f} m')
print(f'  CEP50  : {test_errors.quantile(0.50):.1f} m')
print(f'  CEP90  : {test_errors.quantile(0.90):.1f} m')

Coordinate bounds:
  est_x: [-4095.1, 4096.2]  | out-of-bounds: 3345
  est_y: [-3381.0, 4148.2]  | out-of-bounds: 3519

NaN counts:
est_x     0
est_y     0
true_x    0
true_y    0

Row count: 7919 (expected 7919)  OK=True

Test error distribution (K=5):
  Mean   : 502.1 m
  Median : 174.7 m
  CEP50  : 174.7 m
  CEP90  : 1585.9 m


## 7. Save Outputs

Persist `mr_located` to `data/processed/mr_located.parquet` for consumption by M3 (`evaluate.py`) and M4 (`visualize.py`).

| Output file | Rows | Columns | Consumer |
|---|---|---|---|
| `data/processed/mr_located.parquet` | n_events | 7 | M3 — metrics, M4 — maps |

In [8]:
out_path = f'{PROCESSED_DIR}/mr_located.parquet'
mr_located.to_parquet(out_path, index=False)
print(f'Saved mr_located -> {out_path}  ({len(mr_located):,} rows x {mr_located.shape[1]} cols)')
print(f'Columns: {list(mr_located.columns)}')

Saved mr_located -> ../data/processed/mr_located.parquet  (7,919 rows x 8 cols)
Columns: ['ue_id', 'date', 'true_x', 'true_y', 'est_x', 'est_y', 'split', 'error_m']
